In [21]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns

# make AG2/paths.py importable (this notebook lives in AG2/analysis/)
sys.path.insert(0, str(Path('..').resolve()))
import paths

RUN_ID = "baseline_olympiad_gpt41_n50_20260609"
RUN_DIR = paths.run_dir(RUN_ID)

RESULTS_DIR = RUN_DIR / paths.JUDGE_SUBDIR
OUTPUTS_DIR = RUN_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [22]:
# Load predictions and metadata
pred = pd.read_csv(RESULTS_DIR / "predictions.csv")
summary = pd.read_csv(RESULTS_DIR / "summary.csv")

with open(RUN_DIR / "parsed_traces.json") as f:
    parsed = json.load(f)

meta = pd.DataFrame([
    {"trace_id": t["metadata"]["trace_id"], "correct": t["metadata"]["correct"]}
    for t in parsed
])

df = pred.merge(meta, on="trace_id", how="left")

FM_CODES = ["1.1","1.2","1.3","1.4","1.5",
            "2.1","2.2","2.3","2.4","2.5","2.6",
            "3.1","3.2","3.3"]

FM_NAMES = {
    "1.1": "Disobey Task Specification",
    "1.2": "Disobey Role Specification",
    "1.3": "Step Repetition",
    "1.4": "Loss of Conversation History",
    "1.5": "Unaware of Termination Conditions",
    "2.1": "Conversation Reset",
    "2.2": "Fail to Ask for Clarification",
    "2.3": "Task Derailment",
    "2.4": "Information Withholding",
    "2.5": "Ignored Other Agent's Input",
    "2.6": "Action-Reasoning Mismatch",
    "3.1": "Premature Termination",
    "3.2": "No or Incorrect Verification",
    "3.3": "Weak Verification",
}

n_traces = len(df)
print(f"Loaded {n_traces} traces  |  correct: {df['correct'].sum()}  |  incorrect: {(~df['correct']).sum()}")
summary

Loaded 49 traces  |  correct: 32  |  incorrect: 17


,name,model,total_cost_usd,mean_cost_usd,total_latency_s,mean_latency_s
0,olympiad_gemini25flash_zero_shot,gemini-2.5-flash,0.0,0.0,699.631726,14.278198


## Baseline FM Statistics Table

In [23]:
N_BOOT     = 10_000
SEED       = 42
TARGET_FMS = {"1.1", "3.3", "2.6"}

fail_arr = (~df["correct"].values).astype(float)
rng = np.random.default_rng(SEED)
idx = rng.integers(0, n_traces, size=(N_BOOT, n_traces))

def _sgn(v):
    return f"+{v:.1f}" if v >= 0 else f"{v:.1f}"

rows = []
for code in FM_CODES:
    fm_vals   = df[code].values.astype(float)
    n_present = int(fm_vals.sum())

    if n_present == 0:
        rows.append({
            "FM":                        f"FM {code}  {FM_NAMES[code]}",
            "n":                         0,
            "Prevalence % [95% CI]":     "0.0 [0.0, 0.0]",
            "Failure rate FM+":          "-",
            "Failure rate FM-":          f"{fail_arr.mean()*100:.1f}%",
            "Failure delta pp [95% CI]": "-",
            "_target":                   code in TARGET_FMS,
            "_sort":                     float("nan"),
        })
        continue

    prevalence = fm_vals.mean() * 100
    boot_prev  = fm_vals[idx].mean(axis=1) * 100
    prev_lo    = float(np.percentile(boot_prev, 2.5))
    prev_hi    = float(np.percentile(boot_prev, 97.5))

    with_fm    = fail_arr[fm_vals == 1]
    without_fm = fail_arr[fm_vals == 0]
    fail_with    = with_fm.mean()    if len(with_fm)    > 0 else float("nan")
    fail_without = without_fm.mean() if len(without_fm) > 0 else float("nan")
    delta = (fail_with - fail_without) * 100

    boot_deltas = np.empty(N_BOOT)
    for i in range(N_BOOT):
        samp_fm   = fm_vals[idx[i]]
        samp_fail = fail_arr[idx[i]]
        w  = samp_fail[samp_fm == 1]
        wo = samp_fail[samp_fm == 0]
        if len(w) == 0 or len(wo) == 0:
            boot_deltas[i] = np.nan
        else:
            boot_deltas[i] = (w.mean() - wo.mean()) * 100
    delta_lo = float(np.nanpercentile(boot_deltas, 2.5))
    delta_hi = float(np.nanpercentile(boot_deltas, 97.5))

    prev_str  = f"{prevalence:.1f} [{prev_lo:.1f}, {prev_hi:.1f}]"
    delta_str = (
        f"{_sgn(delta)} *"
        if n_present == 1
        else f"{_sgn(delta)} [{_sgn(delta_lo)}, {_sgn(delta_hi)}]"
    )

    rows.append({
        "FM":                        f"FM {code}  {FM_NAMES[code]}",
        "n":                         n_present,
        "Prevalence % [95% CI]":     prev_str,
        "Failure rate FM+":          f"{fail_with*100:.1f}%" if not np.isnan(fail_with) else "-",
        "Failure rate FM-":          f"{fail_without*100:.1f}%" if not np.isnan(fail_without) else "-",
        "Failure delta pp [95% CI]": delta_str,
        "_target":                   code in TARGET_FMS,
        "_sort":                     delta,
    })

fm_table = (
    pd.DataFrame(rows)
    .sort_values("_sort", ascending=False, na_position="last")
    .reset_index(drop=True)
)

VIS_COLS = [
    "FM", "n",
    "Prevalence % [95% CI]",
    "Failure rate FM+", "Failure rate FM-",
    "Failure delta pp [95% CI]",
]

fm_table[VIS_COLS].to_csv(OUTPUTS_DIR / "baseline_fm_table.csv", index=False)
print(f"Saved → {OUTPUTS_DIR / 'baseline_fm_table.csv'}\n")
print(fm_table[VIS_COLS].to_string(index=False))
print("\n* n = 1; failure delta estimate unreliable")

def _bold_targets(row):
    style = "font-weight: bold" if row["_target"] else ""
    return [style] * len(row)

display(
    fm_table[VIS_COLS + ["_target"]]
    .style
    .apply(_bold_targets, axis=1)
    .hide(axis="index")
    .hide(["_target"], axis="columns")
    .set_caption(
        f"Baseline FM Statistics  (n = {n_traces}; "
        "* n = 1, failure delta estimate unreliable; "
        "bold = target FM of a Stage 2 intervention)"
    )
    .set_properties(**{"text-align": "left"},  subset=["FM"])
    .set_properties(**{"text-align": "right"}, subset=[
        "n", "Prevalence % [95% CI]",
        "Failure rate FM+", "Failure rate FM-",
        "Failure delta pp [95% CI]",
    ])
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "1.0em"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th",
         "props": [("text-align", "center"), ("background-color", "#f5f5f5")]},
    ])
)

Saved → C:\Users\bramn\Desktop\Thesis\agent-evaluation-framework\Bram\AG2\results\baseline_olympiad_gpt41_n50_20260609\outputs\baseline_fm_table.csv

                                       FM  n Prevalence % [95% CI] Failure rate FM+ Failure rate FM- Failure delta pp [95% CI]
            FM 3.1  Premature Termination  1        2.0 [0.0, 6.1]           100.0%            33.3%                   +66.7 *
     FM 1.4  Loss of Conversation History  1        2.0 [0.0, 6.1]           100.0%            33.3%                   +66.7 *
                  FM 1.3  Step Repetition  4       8.2 [2.0, 16.3]            75.0%            31.1%      +43.9 [-27.1, +78.7]
       FM 1.1  Disobey Task Specification 14     28.6 [16.3, 40.8]            57.1%            25.7%       +31.4 [+0.9, +60.8]
                FM 3.3  Weak Verification  9      18.4 [8.2, 30.6]            55.6%            30.0%      +25.6 [-11.6, +62.8]
                  FM 2.3  Task Derailment  4       8.2 [2.0, 16.3]            50.0%     

FM,n,Prevalence % [95% CI],Failure rate FM+,Failure rate FM-,Failure delta pp [95% CI]
FM 3.1 Premature Termination,1,"2.0 [0.0, 6.1]",100.0%,33.3%,+66.7 *
FM 1.4 Loss of Conversation History,1,"2.0 [0.0, 6.1]",100.0%,33.3%,+66.7 *
FM 1.3 Step Repetition,4,"8.2 [2.0, 16.3]",75.0%,31.1%,"+43.9 [-27.1, +78.7]"
FM 1.1 Disobey Task Specification,14,"28.6 [16.3, 40.8]",57.1%,25.7%,"+31.4 [+0.9, +60.8]"
FM 3.3 Weak Verification,9,"18.4 [8.2, 30.6]",55.6%,30.0%,"+25.6 [-11.6, +62.8]"
FM 2.3 Task Derailment,4,"8.2 [2.0, 16.3]",50.0%,33.3%,"+16.7 [-38.3, +72.3]"
FM 2.6 Action-Reasoning Mismatch,9,"18.4 [8.2, 30.6]",44.4%,32.5%,"+11.9 [-25.1, +50.0]"
FM 2.2 Fail to Ask for Clarification,3,"6.1 [0.0, 14.3]",33.3%,34.8%,"-1.4 [-44.7, +69.6]"
FM 3.2 No or Incorrect Verification,3,"6.1 [0.0, 14.3]",33.3%,34.8%,"-1.4 [-44.7, +68.8]"
FM 1.2 Disobey Role Specification,1,"2.0 [0.0, 6.1]",0.0%,35.4%,-35.4 *


In [24]:
fm_slide = (
    fm_table[VIS_COLS + ["_target", "_sort"]]
    .sort_values(["n", "_sort"], ascending=[False, False], na_position="last")
    .reset_index(drop=True)
)

footnote = (
    f"Baseline FM Statistics  (n = {n_traces} traces; "
    "sorted by n descending, then failure delta descending; "
    "bold = target FM of a Stage 2 intervention)"
)

display(
    fm_slide[VIS_COLS + ["_target"]]
    .style
    .apply(lambda row: (
        ["font-weight: bold"] * len(row) if row["_target"] else [""] * len(row)
    ), axis=1)
    .hide(axis="index")
    .hide(["_target"], axis="columns")
    .set_caption(footnote)
    .set_properties(**{"text-align": "left"},  subset=["FM"])
    .set_properties(**{"text-align": "right"}, subset=[
        "n", "Prevalence % [95% CI]",
        "Failure rate FM+", "Failure rate FM-",
        "Failure delta pp [95% CI]",
    ])
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "1.05em"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th",
         "props": [("text-align", "center"), ("background-color", "#f5f5f5")]},
    ])
)

FM,n,Prevalence % [95% CI],Failure rate FM+,Failure rate FM-,Failure delta pp [95% CI]
FM 1.1 Disobey Task Specification,14,"28.6 [16.3, 40.8]",57.1%,25.7%,"+31.4 [+0.9, +60.8]"
FM 3.3 Weak Verification,9,"18.4 [8.2, 30.6]",55.6%,30.0%,"+25.6 [-11.6, +62.8]"
FM 2.6 Action-Reasoning Mismatch,9,"18.4 [8.2, 30.6]",44.4%,32.5%,"+11.9 [-25.1, +50.0]"
FM 1.3 Step Repetition,4,"8.2 [2.0, 16.3]",75.0%,31.1%,"+43.9 [-27.1, +78.7]"
FM 2.3 Task Derailment,4,"8.2 [2.0, 16.3]",50.0%,33.3%,"+16.7 [-38.3, +72.3]"
FM 2.2 Fail to Ask for Clarification,3,"6.1 [0.0, 14.3]",33.3%,34.8%,"-1.4 [-44.7, +69.6]"
FM 3.2 No or Incorrect Verification,3,"6.1 [0.0, 14.3]",33.3%,34.8%,"-1.4 [-44.7, +68.8]"
FM 3.1 Premature Termination,1,"2.0 [0.0, 6.1]",100.0%,33.3%,+66.7 *
FM 1.4 Loss of Conversation History,1,"2.0 [0.0, 6.1]",100.0%,33.3%,+66.7 *
FM 1.2 Disobey Role Specification,1,"2.0 [0.0, 6.1]",0.0%,35.4%,-35.4 *
